# Amazon Review Analyzer — Exploratory Data Analysis

This notebook explores the Amazon Fine Food Reviews dataset, examines class distributions,
sentiment patterns, and validates the feature engineering pipeline.

**Dataset:** [Amazon Fine Food Reviews — Kaggle](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.insert(0, os.path.abspath('..'))
from model.features import clean_text, vader_score, label_from_rating, sentiment_label

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
print('Imports OK')

## 1. Load Data

In [ ]:
# Use the sample if the full dataset is not available
DATA_PATH = '../data/reviews_sample.csv'
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f'Run: python data/generate_sample.py  OR  place Reviews.csv in data/')

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['Text', 'Score'])
df['Score'] = pd.to_numeric(df['Score'], errors='coerce').clip(1, 5)
df['label'] = df['Score'].apply(label_from_rating)

print(f'Shape: {df.shape}')
df.head()

## 2. Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Star rating histogram
df['Score'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Star Rating Distribution')
axes[0].set_xlabel('Stars'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Class label pie
df['label'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
    colors=['#4caf50', '#ff9800', '#f44336'], startangle=90)
axes[1].set_title('Class Label Distribution')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../assets/rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(df['label'].value_counts())

## 3. Sentiment Analysis with VADER

In [ ]:
print('Computing VADER scores (may take a moment)...')
df['sentiment'] = df['Text'].apply(vader_score)
df['sentiment_label'] = df['sentiment'].apply(sentiment_label)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sentiment distribution by label
for label, color in zip(['Buy', 'Caution', "Don't Buy"], ['#4caf50', '#ff9800', '#f44336']):
    subset = df[df['label'] == label]['sentiment']
    axes[0].hist(subset, bins=30, alpha=0.6, label=label, color=color)
axes[0].set_title('VADER Sentiment by Label')
axes[0].set_xlabel('Compound Score'); axes[0].legend()

# Avg sentiment per star rating
df.groupby('Score')['sentiment'].mean().plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Avg VADER Sentiment per Star Rating')
axes[1].set_xlabel('Stars'); axes[1].set_ylabel('Avg Compound Score')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../assets/sentiment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Review Length Analysis

In [ ]:
df['review_length'] = df['Text'].apply(lambda x: len(str(x).split()))

fig, ax = plt.subplots(figsize=(10, 4))
for label, color in zip(['Buy', 'Caution', "Don't Buy"], ['#4caf50', '#ff9800', '#f44336']):
    subset = df[df['label'] == label]['review_length'].clip(0, 300)
    ax.hist(subset, bins=40, alpha=0.6, label=label, color=color)
ax.set_title('Review Length Distribution by Label')
ax.set_xlabel('Word Count'); ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('../assets/review_length.png', dpi=150, bbox_inches='tight')
plt.show()

print(df.groupby('label')['review_length'].describe())

## 5. Most Common Words per Class

In [ ]:
from collections import Counter

df['clean'] = df['Text'].apply(clean_text)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, label in zip(axes, ['Buy', 'Caution', "Don't Buy"]):
    words = ' '.join(df[df['label'] == label]['clean']).split()
    top = Counter(words).most_common(15)
    words_l, counts = zip(*top)
    ax.barh(words_l[::-1], counts[::-1], color='steelblue')
    ax.set_title(f'Top words — {label}')
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('../assets/top_words.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlation: Sentiment vs Star Rating

In [ ]:
corr = df[['Score', 'sentiment', 'review_length']].corr()
plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../assets/correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Findings

| Observation | Implication |
|---|---|
| Dataset is skewed towards 5-star reviews | Use `class_weight='balanced'` in classifier |
| VADER compound score strongly correlates with star rating | Good feature for the model |
| Negative reviews tend to be longer | Review length is a useful feature |
| Distinct vocabulary per class | TF-IDF will capture discriminative n-grams |
